# 03 - Model inference

We now have our semantic axis and our generated dataset, which gives us several image sets to work with:

- the **original** ImageNet images,
- the **background-only** images, from which the object was removed by inpainting,
- the **object-only** images, where the object is shown on a black, white or pink-noise background,
- the **object library** with its object-only versions,
- and the **composites**, where object cut-outs are placed on backgrounds along the semantic axis.

In this notebook, we check how different models classify these images. For every model and image set, we save the top-5 predictions as a CSV file, which the following notebooks use for the evaluation.

In [ ]:
import os
from pathlib import Path

# Repository paths
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
CLASS_INDEX = REPO_ROOT / "helper_files" / "imagenet_class_index.json"

# Where the prediction CSVs are written
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"

# ImageNet-OOC1k dataset: provided images and metadata in source/, everything we generate in generated/
DATASET_DIR = REPO_ROOT / "ImageNet-OOC1k"
SOURCE_DIR = DATASET_DIR / "source"
GENERATED_DIR = DATASET_DIR / "generated"
COMPOSITES_DIR = GENERATED_DIR / "composites"

# Model weight caches (set these before importing timm). None keeps the default caches in your
# home folder; set a path if you want the weights on a bigger disk.
CACHE_DIR = None
if CACHE_DIR is not None:
    os.environ["HF_HOME"] = str(CACHE_DIR / "hf_cache")
    os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch_cache")

# The model weights are downloaded from the Hugging Face hub. If a download asks you
# to log in, add your Hugging Face token here.
HF_TOKEN = None
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

In [2]:
# Inference functions live outside the notebook to keep it readable.
from ooc1k.inference.model_inference import (
    CLIP_MODEL,
    load_ground_truth_composites,
    load_ground_truth_library,
    load_ground_truth_main,
    run_inference,
)

/pfss/mlde/workspaces/mlde_wsp_PI_Roig/bersch/conda/envs/ooc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Image conditions

Each image condition points to the folder (or, for the composites, the HDF5 file) that holds its images, together with the ground truth, which maps every image name to its ImageNet class. For the composites, the ground truth is the class of the placed object.

The main images (originals, background-only and object-only) share one ground truth, the object library has its own, and so do the composites.

In [ ]:
gt_main = load_ground_truth_main(SOURCE_DIR / "metadata_backgrounds.csv", CLASS_INDEX)
gt_library = load_ground_truth_library(SOURCE_DIR / "metadata_objects.csv")
gt_composites = load_ground_truth_composites(COMPOSITES_DIR / "metadata.csv", CLASS_INDEX)

OBJECT_ONLY_DIR = GENERATED_DIR / "object_only"

CONDITIONS = {
    # Main images
    # "original_main": (SOURCE_DIR / "originals", gt_main),
    # "background_only": (SOURCE_DIR / "backgrounds", gt_main),
    "object_main_black": (OBJECT_ONLY_DIR / "main" / "black", gt_main),
    "object_main_white": (OBJECT_ONLY_DIR / "main" / "white", gt_main),
    "object_main_pink_noise": (OBJECT_ONLY_DIR / "main" / "pink_noise", gt_main),
    # Object library
    "original_library": (SOURCE_DIR / "object_library" / "images", gt_library),
    "object_library_black": (OBJECT_ONLY_DIR / "library" / "black", gt_library),
    "object_library_white": (OBJECT_ONLY_DIR / "library" / "white", gt_library),
    "object_library_pink_noise": (OBJECT_ONLY_DIR / "library" / "pink_noise", gt_library),
    # Composites
    # "composite_ooc": (COMPOSITES_DIR / "composites.h5", gt_composites),
}

## Models

We use two sets of models. The **main models** are the six models shown in the main figures of the paper. The **additional models** complete the 35 models of Table 2, covering ViTs, DeiTs, BEiTs, CNNs, EfficientNets and ConvNeXts.

The keys are the names used in the CSV files, the values are the pretrained timm models. CLIP is not part of timm; it is evaluated zero-shot by comparing each image with the text prompt `"a photo of a {class}"` for every ImageNet class.

In [4]:
MAIN_MODELS = {
    "ViT-B-16": "vit_base_patch16_224.augreg2_in21k_ft_in1k",
    "ViT-L-16": "vit_large_patch16_224.augreg_in21k_ft_in1k",
    "DeiT-B-16": "deit_base_patch16_224.fb_in1k",
    "ResNet-50": "resnet50.a1_in1k",
    "EfficientNet-B3": "efficientnet_b3.ra2_in1k",
    "CLIP-ViT-B-16": CLIP_MODEL,
}

ADDITIONAL_MODELS = {
    # ViT
    "vit_small_patch16_224": "vit_small_patch16_224.augreg_in21k_ft_in1k",
    "vit_base_patch32_224": "vit_base_patch32_224.augreg_in21k_ft_in1k",
    "vit_base_patch16_384": "vit_base_patch16_384.augreg_in21k_ft_in1k",
    # DeiT
    "deit_small_patch16_224": "deit_small_patch16_224.fb_in1k",
    "deit3_small_patch16_224": "deit3_small_patch16_224.fb_in1k",
    "deit3_medium_patch16_224": "deit3_medium_patch16_224.fb_in1k",
    "deit3_base_patch16_224": "deit3_base_patch16_224.fb_in1k",
    "deit3_large_patch16_224": "deit3_large_patch16_224.fb_in1k",
    # BEiT
    "beit_base_patch16_224": "beit_base_patch16_224.in22k_ft_in22k_in1k",
    "beit_large_patch16_224": "beit_large_patch16_224.in22k_ft_in22k_in1k",
    "beitv2_base_patch16_224": "beitv2_base_patch16_224.in1k_ft_in22k_in1k",
    "beitv2_large_patch16_224": "beitv2_large_patch16_224.in1k_ft_in22k_in1k",
    "beit3_base_patch16_224": "beit3_base_patch16_224.in22k_ft_in1k",
    "beit3_large_patch16_224": "beit3_large_patch16_224.in22k_ft_in1k",
    # CNN
    "resnet34": "resnet34.a1_in1k",
    "resnet101": "resnet101.a1h_in1k",
    "ssl_resnet50": "resnet50.fb_ssl_yfcc100m_ft_in1k",
    "inception_v3": "inception_v3.tv_in1k",
    "inception_v4": "inception_v4.tf_in1k",
    # EfficientNet
    "efficientnet_b2": "efficientnet_b2.ra_in1k",
    "efficientnet_b4": "efficientnet_b4.ra2_in1k",
    "efficientnetv2_rw_s": "efficientnetv2_rw_s.ra2_in1k",
    "efficientnetv2_rw_m": "efficientnetv2_rw_m.agc_in1k",
    # ConvNeXt
    "convnext_small": "convnext_small.in12k_ft_in1k",
    "convnext_base": "convnext_base.fb_in22k_ft_in1k",
    "convnext_large": "convnext_large.fb_in22k_ft_in1k",
    "convnext_xlarge": "convnext_xlarge.fb_in22k_ft_in1k",
    "convnextv2_base": "convnextv2_base.fcmae_ft_in22k_in1k",
    "convnextv2_large": "convnextv2_large.fcmae_ft_in22k_in1k",
}

ALL_MODELS = MAIN_MODELS | ADDITIONAL_MODELS

## Running the inference

Select the models and conditions you want to run. For every pair, the predictions are saved as `{model}_{condition}_{timestamp}.csv` in `PREDICTIONS_DIR`.

Inference on the composites takes much longer than on the other conditions (about 965,000 images per model). Pairs that already have a CSV are skipped if `SKIP_EXISTING=TRUE`, so you can stop the notebook and continue later. If you already have the prediction CSVs of the paper, point `PREDICTIONS_DIR` to them and only the missing pairs are run.

In [6]:
SELECTED_MODELS = MAIN_MODELS                # MAIN_MODELS, ADDITIONAL_MODELS or ALL_MODELS
SELECTED_CONDITIONS = list(CONDITIONS)       # e.g. ["background_only", "original_main"]
BATCH_SIZE = 128
NUM_WORKERS = 4
SKIP_EXISTING = False

run_inference(
    models=SELECTED_MODELS,
    conditions={name: CONDITIONS[name] for name in SELECTED_CONDITIONS},
    class_index_path=CLASS_INDEX,
    output_dir=PREDICTIONS_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    skip_existing=SKIP_EXISTING,
)

print(f"Saved to {PREDICTIONS_DIR}")

ViT-B-16 | object_main_black: top-1 accuracy 73.1% on 1,137 images


ViT-B-16 | object_main_white: top-1 accuracy 75.4% on 1,137 images


ViT-B-16 | object_main_pink_noise: top-1 accuracy 70.5% on 1,137 images


ViT-B-16 | original_library: top-1 accuracy 86.5% on 12,358 images


ViT-B-16 | object_library_black: top-1 accuracy 80.5% on 12,358 images


ViT-B-16 | object_library_white: top-1 accuracy 80.9% on 12,358 images


... (32 outputs left out)

CLIP-ViT-B-16 | original_library: top-1 accuracy 66.1% on 12,358 images


CLIP-ViT-B-16 | object_library_black: top-1 accuracy 57.7% on 12,358 images


CLIP-ViT-B-16 | object_library_white: top-1 accuracy 57.9% on 12,358 images


CLIP-ViT-B-16 | object_library_pink_noise: top-1 accuracy 55.7% on 12,358 images
Saved to /pfss/mlde/workspaces/mlde_wsp_PI_Roig/bersch/repositories/OOC-Dataset-Creator/imagenet-ooc1k/outputs/predictions
